# BraTS 2024 — ABMIL + U-Net Evaluation Notebook

This notebook evaluates the **ABMIL + U-Net** model trained in `abmil-unet-brats2024.ipynb`.

### Key differences from the original evaluation notebook:
- Loads the saved **patch-based U-Net** (`model_abmil_unet_brats2024.keras`)
- Uses **ABMIL** to select top-K suspicious patches per patient before feeding U-Net
- Reconstructs the full 3D prediction volume from patch predictions
- All Dice, HD95, Tumor Core, and Whole Tumor metrics are computed on the **reconstructed full volume**
- Visualization shows ABMIL attention weights + final segmentation


## Cell 1 — Imports

In [ ]:
import os
import gc
import numpy as np
import nibabel as nib
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from scipy.ndimage import distance_transform_edt, binary_erosion
from sklearn.utils import shuffle
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K

print("TensorFlow :", tf.__version__)
print("PyTorch    :", torch.__version__)


## Cell 2 — NIfTI File Loader

In [ ]:
def load_nii(path, is_mask=False):
    """Load a .nii file → numpy array (128, 128, 128)."""
    nii = nib.load(path)
    arr = np.array(nii.get_fdata(), dtype=np.float32)
    if is_mask:
        arr = arr.astype(np.int32)
    return arr


## Cell 3 — ABMIL Model Definition (must match training exactly)

We need to re-define the ABMIL architecture so we can load the trained weights.
These class definitions are **identical** to the training notebook.


In [ ]:
class PatchFeatureExtractor(nn.Module):
    def __init__(self, in_channels=4, out_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128), nn.ReLU(), nn.AdaptiveAvgPool3d(1),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


class PatchAttention(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, 64), nn.Tanh(), nn.Linear(64, 1)
        )

    def forward(self, patch_features):
        scores  = self.attention(patch_features)          # (1, 64, 1)
        weights = torch.softmax(scores, dim=1)            # sum to 1
        bag_feature = torch.sum(weights * patch_features, dim=1)
        return bag_feature, weights


class ABMIL(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.feature_extractor = PatchFeatureExtractor(in_channels=4, out_dim=feature_dim)
        self.patch_attention    = PatchAttention(feature_dim)

    def forward(self, patches):
        N     = patches.shape[0]
        feats = self.feature_extractor(patches)   # (N, 128)
        feats = feats.unsqueeze(0)                # (1, N, 128)
        bag_feature, weights = self.patch_attention(feats)
        weights = weights.squeeze(0).squeeze(-1)  # (N,)
        return weights

print("ABMIL classes defined.")


## Cell 4 — Patch Extraction & Reconstruction Utilities

Two functions:
- `extract_patches` — cuts the 128³ volume into 64 patches of 32³ (same as training)
- `reconstruct_volume` — **new** — stitches 64 patch predictions back into one 128³ volume


In [ ]:
def extract_patches(volume, mask, patch_size=32):
    """
    Cut brain volume into non-overlapping 3D patches.
    volume : (128, 128, 128, 4)
    mask   : (128, 128, 128, 5)
    Returns patches (64, 32, 32, 32, 4), mask_patches (64, 32, 32, 32, 5)
    """
    D, H, W, C = volume.shape
    P = patch_size
    patches, mask_patches = [], []
    for d in range(0, D, P):
        for h in range(0, H, P):
            for w in range(0, W, P):
                patches.append(volume[d:d+P, h:h+P, w:w+P, :])
                mask_patches.append(mask[d:d+P, h:h+P, w:w+P, :])
    return np.array(patches), np.array(mask_patches)


def reconstruct_volume(patch_preds, vol_shape=(128, 128, 128),
                        patch_size=32, num_classes=5):
    """
    Stitch patch predictions back into a full 128×128×128 label volume.

    patch_preds : dict  {patch_index: np.array (32,32,32,5)}
                  Only predicted patches — unpredicted patches default to class 0.
    Returns     : np.array (128, 128, 128) with hard class labels
    """
    full_softmax = np.zeros((*vol_shape, num_classes), dtype=np.float32)
    # mark background as confident 1.0 everywhere first
    full_softmax[..., 0] = 1.0

    patch_idx = 0
    P = patch_size
    for d in range(0, vol_shape[0], P):
        for h in range(0, vol_shape[1], P):
            for w in range(0, vol_shape[2], P):
                if patch_idx in patch_preds:
                    full_softmax[d:d+P, h:h+P, w:w+P, :] = patch_preds[patch_idx]
                patch_idx += 1

    return np.argmax(full_softmax, axis=-1)   # (128, 128, 128)

print("Patch utilities defined.")


## Cell 5 — ABMIL Patch Selector

In [ ]:
def select_top_patches(patches, mask_patches, abmil_model, top_k=10, device='cpu'):
    """
    Use ABMIL to score all 64 patches and return the top_k most suspicious.
    Returns top_patches, top_mask_patches, top_indices, all_weights
    """
    abmil_model.eval()
    patches_torch = torch.tensor(
        patches.transpose(0, 4, 1, 2, 3), dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        weights = abmil_model(patches_torch)

    weights_np  = weights.cpu().numpy()
    top_indices = np.argsort(weights_np)[::-1][:top_k]

    return (patches[top_indices],
            mask_patches[top_indices],
            top_indices,
            weights_np)

print("Patch selector defined.")


## Cell 6 — Dice & HD95 Metric Helpers

In [ ]:
# ── Per-class Dice (used for per-class breakdown) ──────────────────────────
def dice_coef_class(y_true, y_pred, class_index, epsilon=1e-6):
    inter = K.sum(K.abs(y_true[..., class_index] * y_pred[..., class_index]))
    denom = (K.sum(K.square(y_true[..., class_index])) +
             K.sum(K.square(y_pred[..., class_index])) + epsilon)
    return (2. * inter) / denom


# ── Mean Dice over all classes ─────────────────────────────────────────────
def dice_coef_whole(y_true, y_pred, epsilon=1e-6):
    num_classes = y_true.shape[-1]
    total = 0
    for i in range(num_classes):
        yt = K.flatten(y_true[..., i])
        yp = K.flatten(y_pred[..., i])
        inter = K.sum(yt * yp)
        total += (2. * inter) / (K.sum(K.square(yt)) + K.sum(K.square(yp)) + epsilon)
    return total / num_classes


# ── Tumor Core Dice: ET(1) + NETC(2) ──────────────────────────────────────
@tf.function
def diceTumorCore(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    yt = y_true[..., 1] + y_true[..., 2]
    yp = y_pred[..., 1] + y_pred[..., 2]
    return (2.*tf.reduce_sum(yt*yp)+smooth) / (tf.reduce_sum(yt+yp)+smooth)


# ── Whole Tumor Dice: ET(1)+NETC(2)+SNFH(3) ───────────────────────────────
@tf.function
def diceWholeTumor(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    yt = y_true[..., 1] + y_true[..., 2] + y_true[..., 3]
    yp = y_pred[..., 1] + y_pred[..., 2] + y_pred[..., 3]
    return (2.*tf.reduce_sum(yt*yp)+smooth) / (tf.reduce_sum(yt+yp)+smooth)


# ── HD95 ───────────────────────────────────────────────────────────────────
def hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if pred.sum() == 0 and gt.sum() == 0:
        return 0.0
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    surface_pred = pred ^ binary_erosion(pred)
    surface_gt   = gt   ^ binary_erosion(gt)
    if surface_pred.sum() == 0 or surface_gt.sum() == 0:
        return np.nan
    dt_pred = distance_transform_edt(~surface_pred)
    dt_gt   = distance_transform_edt(~surface_gt)
    all_d = np.concatenate([dt_gt[surface_pred], dt_pred[surface_gt]])
    return np.percentile(all_d, 95) if all_d.size > 0 else np.nan

print("Metric helpers defined.")


## Cell 7 — Dataset Path & Patient List

In [ ]:
# ── EDIT THIS PATH to point at your test/validation data ──────────────────
root_dir = '/kaggle/input/datasets/muhammadwahajsajid/brats-2024-dataset-tumor-centered-cropped/BraTs 2024 Data Tumor Centered Cropped/Training Data'

all_patients = sorted(os.listdir(root_dir))
print(f"Total patients found: {len(all_patients)}")
print(f"Example IDs: {all_patients[:3]}")


## Cell 8 — Load ABMIL (PyTorch) + U-Net (Keras)

- `abmil_model` — loaded from the `.pt` checkpoint saved during training
- `unet_model`  — loaded from `model_abmil_unet_brats2024.keras`


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# ── Load ABMIL weights ──────────────────────────────────────────────────────
# If you saved ABMIL state dict during training, load it here.
# If you did NOT save ABMIL separately, it will run with random weights
# (patch selection will still work but be unoptimised).
ABMIL_WEIGHTS_PATH = 'checkpoints/abmil_weights.pt'   # ← change if needed

abmil_model = ABMIL(feature_dim=128).to(DEVICE)

if os.path.exists(ABMIL_WEIGHTS_PATH):
    abmil_model.load_state_dict(torch.load(ABMIL_WEIGHTS_PATH, map_location=DEVICE))
    print(f"✅ ABMIL weights loaded from: {ABMIL_WEIGHTS_PATH}")
else:
    print(f"⚠️  ABMIL weights file not found at: {ABMIL_WEIGHTS_PATH}")
    print("   Running with untrained ABMIL weights (patch selection is random-weighted).")
    print("   To save ABMIL weights during training add:  torch.save(abmil_model.state_dict(), 'checkpoints/abmil_weights.pt')")

abmil_model.eval()

# ── Load U-Net ──────────────────────────────────────────────────────────────
UNET_PATH = 'model_abmil_unet_brats2024.keras'   # ← change if needed

unet_model = tf.keras.models.load_model(
    UNET_PATH,
    custom_objects={
        'total_loss'             : lambda yt, yp: (1 - dice_coef_whole(yt, yp)) + tf.keras.losses.categorical_crossentropy(yt, yp),
        'dice_coef'              : dice_coef_whole,
        'dice_coef_whole'        : dice_coef_whole,
        'DicePerClass'           : tf.keras.metrics.MeanMetricWrapper,   # placeholder
        'DiceWholeTumorMetric'   : tf.keras.metrics.MeanMetricWrapper,   # placeholder
    },
    compile=False,
    safe_mode=False
)
print(f"✅ U-Net loaded from: {UNET_PATH}")
print(f"   Input  shape : {unet_model.input_shape}")
print(f"   Output shape : {unet_model.output_shape}")


## Cell 9 — Per-Patient Inference Function

This is the core evaluation function. For each patient it:
1. Loads all 4 MRI modalities + ground truth mask
2. Normalises and stacks them → `(128,128,128,4)`
3. Cuts into 64 patches → ABMIL scores them → keeps top-K
4. U-Net predicts each top patch → softmax `(32,32,32,5)`
5. **Reconstructs** a full `(128,128,128)` label volume from patch predictions
6. Returns the full prediction and ground truth for metric computation


In [ ]:
def predict_patient_abmil_unet(root_dir, patient_id,
                                abmil_model, unet_model,
                                patch_size=32, top_k=10,
                                device='cpu'):
    """
    Full ABMIL + U-Net inference for one patient.

    Returns
    -------
    pred_vol   : np.array (128,128,128)  hard class labels (0-4)
    true_vol   : np.array (128,128,128)  ground truth class labels (0-4)
    weights_np : np.array (64,)          ABMIL attention weight per patch
    top_indices: np.array (top_k,)       which patch indices were selected
    volume     : np.array (128,128,128,4) normalised MRI (for visualisation)
    """
    patient_path = os.path.join(root_dir, patient_id)

    # ── Load & normalise 4 modalities ──────────────────────────────────────
    def norm(x):
        return (x - np.mean(x)) / (np.std(x) + 1e-8)

    t1c  = norm(load_nii(os.path.join(patient_path, f"{patient_id}-t1c.nii")))
    t1n  = norm(load_nii(os.path.join(patient_path, f"{patient_id}-t1n.nii")))
    t2f  = norm(load_nii(os.path.join(patient_path, f"{patient_id}-t2f.nii")))
    t2w  = norm(load_nii(os.path.join(patient_path, f"{patient_id}-t2w.nii")))
    mask = load_nii(os.path.join(patient_path, f"{patient_id}-seg.nii"), is_mask=True)

    volume   = np.stack([t1c, t1n, t2f, t2w], axis=-1)  # (128,128,128,4)
    mask_cat = to_categorical(mask, num_classes=5)        # (128,128,128,5)

    # ── Step 1: Extract 64 patches ─────────────────────────────────────────
    patches, mask_patches = extract_patches(volume, mask_cat, patch_size)
    # (64, 32,32,32,4) and (64, 32,32,32,5)

    # ── Step 2: ABMIL → top-K suspicious patches ───────────────────────────
    top_patches, top_masks, top_indices, weights_np = select_top_patches(
        patches, mask_patches, abmil_model, top_k=top_k, device=device
    )

    # ── Step 3: U-Net predicts each top patch ──────────────────────────────
    patch_preds = {}
    unet_preds = unet_model.predict(top_patches, batch_size=top_k, verbose=0)
    # unet_preds: (top_k, 32,32,32,5)

    for i, patch_idx in enumerate(top_indices):
        patch_preds[int(patch_idx)] = unet_preds[i]   # softmax (32,32,32,5)

    # ── Step 4: Reconstruct full 128³ volume ───────────────────────────────
    pred_vol = reconstruct_volume(
        patch_preds,
        vol_shape=(128, 128, 128),
        patch_size=patch_size,
        num_classes=5
    )   # (128,128,128) hard labels

    true_vol = mask.copy()   # already (128,128,128) integer labels

    return pred_vol, true_vol, weights_np, top_indices, volume

print("Per-patient inference function defined.")


## Cell 10 — Training History Plot

In [ ]:
# ── Change this path to your training log CSV ─────────────────────────────
LOG_PATH = 'training_abmil_unet.log'

if os.path.exists(LOG_PATH):
    hist = pd.read_csv(LOG_PATH)
    print("Columns:", list(hist.columns))

    epoch = range(len(hist))
    fig, axes = plt.subplots(1, 4, figsize=(22, 4))

    pairs = [
        ('loss',                'val_loss',                'Loss'),
        ('dice_coef',           'val_dice_coef',           'Overall Dice'),
        ('dice_whole_tumor',    'val_dice_whole_tumor',    'Whole Tumor Dice'),
        ('learning_rate',       None,                      'Learning Rate'),
    ]

    for ax, (train_col, val_col, title) in zip(axes, pairs):
        if train_col in hist.columns:
            ax.plot(epoch, hist[train_col], 'b', label='Train')
        if val_col and val_col in hist.columns:
            ax.plot(epoch, hist[val_col], 'r', label='Val')
        ax.set_title(title)
        ax.legend()

    plt.tight_layout()
    plt.savefig('abmil_unet_training_curves.png', dpi=150)
    plt.show()
    print("Saved: abmil_unet_training_curves.png")
else:
    print(f"Log file not found at: {LOG_PATH}")
    print("Skipping training curve plot.")


## Cell 11 — Full Dataset Evaluation (Dice per class + Whole Tumor + Tumor Core)

Iterates over every patient, runs inference, and accumulates Dice scores.
Memory-safe: each patient is processed and freed before the next.


In [ ]:
PATCH_SIZE = 32
TOP_K      = 10
N_CLASSES  = 5
CLASS_NAMES = ["Background", "ET", "NETC", "SNFH", "RC"]

dice_sums    = np.zeros(N_CLASSES, dtype=np.float64)
dice_tc_list = []
dice_wt_list = []
valid_count  = 0
skipped      = []

print(f"Evaluating {len(all_patients)} patients...")
print("-" * 50)

for i, patient_id in enumerate(all_patients):
    try:
        pred_vol, true_vol, _, _, _ = predict_patient_abmil_unet(
            root_dir, patient_id,
            abmil_model, unet_model,
            patch_size=PATCH_SIZE, top_k=TOP_K, device=DEVICE
        )

        # Convert to one-hot tensors for metric functions
        pred_oh = tf.one_hot(pred_vol, depth=N_CLASSES)   # (128,128,128,5)
        true_oh = tf.one_hot(true_vol, depth=N_CLASSES)   # (128,128,128,5)

        # Add batch dim for metric functions: (1,128,128,128,5)
        pred_oh_b = tf.expand_dims(tf.cast(pred_oh, tf.float32), 0)
        true_oh_b = tf.expand_dims(tf.cast(true_oh, tf.float32), 0)

        # Per-class Dice
        for c in range(N_CLASSES):
            dice_sums[c] += float(dice_coef_class(true_oh_b, pred_oh_b, c).numpy())

        # Tumor Core + Whole Tumor
        dice_tc_list.append(float(diceTumorCore(true_oh_b, pred_oh_b).numpy()))
        dice_wt_list.append(float(diceWholeTumor(true_oh_b, pred_oh_b).numpy()))

        valid_count += 1

        del pred_vol, true_vol, pred_oh, true_oh, pred_oh_b, true_oh_b
        gc.collect()

    except Exception as e:
        print(f"  ⚠️  Skipping {patient_id}: {e}")
        skipped.append(patient_id)

    if (i + 1) % 10 == 0 or (i + 1) == len(all_patients):
        print(f"  Processed {i+1}/{len(all_patients)}")

# ── Results ────────────────────────────────────────────────────────────────
dice_per_class = dice_sums / max(valid_count, 1)

print("\n" + "="*55)
print("  ABMIL + U-Net  —  FULL DATASET DICE EVALUATION")
print("="*55)
for c in range(N_CLASSES):
    print(f"  Dice {CLASS_NAMES[c]:<12}: {dice_per_class[c]:.4f}")
print(f"\n  Tumor Core Dice  (ET + NETC)          : {np.mean(dice_tc_list):.4f}")
print(f"  Whole Tumor Dice (ET + NETC + SNFH)   : {np.mean(dice_wt_list):.4f}")
print(f"\n  Valid patients : {valid_count}")
if skipped:
    print(f"  Skipped        : {len(skipped)} → {skipped}")
print("="*55)


## Cell 12 — HD95 Evaluation (Individual Classes + Composite Regions)

Computes 95th-percentile Hausdorff Distance per class and for Tumor Core / Whole Tumor.
This cell re-runs inference patient-by-patient (memory-safe).


In [ ]:
hd_per_class  = {i: [] for i in range(1, N_CLASSES)}   # skip background (0)
hd_per_region = {"Tumor Core": [], "Whole Tumor": []}
REGION_CLASSES = {"Tumor Core": [1, 2], "Whole Tumor": [1, 2, 3]}

print(f"Computing HD95 for {len(all_patients)} patients...")

for i, patient_id in enumerate(all_patients):
    try:
        pred_vol, true_vol, _, _, _ = predict_patient_abmil_unet(
            root_dir, patient_id,
            abmil_model, unet_model,
            patch_size=PATCH_SIZE, top_k=TOP_K, device=DEVICE
        )

        # Per-class HD95
        for cls in range(1, N_CLASSES):
            hd_val = hd95((pred_vol == cls).astype(np.uint8),
                          (true_vol == cls).astype(np.uint8))
            if hd_val is not None and np.isfinite(hd_val):
                hd_per_class[cls].append(float(hd_val))

        # Composite region HD95
        for region, indices in REGION_CLASSES.items():
            hd_val = hd95(np.isin(pred_vol, indices).astype(np.uint8),
                          np.isin(true_vol, indices).astype(np.uint8))
            if hd_val is not None and np.isfinite(hd_val):
                hd_per_region[region].append(float(hd_val))

        del pred_vol, true_vol
        gc.collect()

    except Exception as e:
        print(f"  ⚠️  Skipping {patient_id}: {e}")

    if (i + 1) % 10 == 0 or (i + 1) == len(all_patients):
        print(f"  Processed {i+1}/{len(all_patients)}")

# ── Print Results ──────────────────────────────────────────────────────────
mean_hd_class  = {c: np.mean(v) if v else np.nan for c, v in hd_per_class.items()}
mean_hd_region = {r: np.mean(v) if v else np.nan for r, v in hd_per_region.items()}
valid_hd = [v for v in mean_hd_class.values() if np.isfinite(v)]

print("\n" + "="*55)
print("  ABMIL + U-Net  —  HD95 EVALUATION")
print("="*55)
print("\n  Individual Classes:")
for cls in range(1, N_CLASSES):
    val = mean_hd_class[cls]
    tag = f"{val:.4f} mm" if np.isfinite(val) else "NaN (no valid samples)"
    print(f"    HD95 {CLASS_NAMES[cls]:<8}: {tag}")

print(f"\n  Mean HD95 (classes 1-4): {np.mean(valid_hd):.4f} mm" if valid_hd else "\n  Mean HD95: NaN")

print("\n  Composite Regions:")
for region, label in [("Tumor Core","ET + NETC"), ("Whole Tumor","ET + NETC + SNFH")]:
    val = mean_hd_region[region]
    tag = f"{val:.4f} mm" if np.isfinite(val) else "NaN"
    print(f"    HD95 {region:<14} ({label}): {tag}")
print("="*55)


## Cell 13 — Visualise One Patient

Shows:
1. ABMIL attention bar chart (which patches were suspicious)
2. 4 MRI modalities + Ground Truth + Prediction side by side


In [ ]:
# ── Pick any patient ID from your list ────────────────────────────────────
VIS_PATIENT = all_patients[0]   # change to e.g. "BraTS-GLI-02318-100"

print(f"Running inference on: {VIS_PATIENT}")

pred_vol, true_vol, weights_np, top_indices, volume = predict_patient_abmil_unet(
    root_dir, VIS_PATIENT,
    abmil_model, unet_model,
    patch_size=PATCH_SIZE, top_k=TOP_K, device=DEVICE
)

SLICE_Z = volume.shape[2] // 2   # middle axial slice

# ── Plot 1: ABMIL attention weights ───────────────────────────────────────
plt.figure(figsize=(14, 4))
plt.bar(range(len(weights_np)), weights_np, color='steelblue', label='All patches')
plt.bar(top_indices, weights_np[top_indices], color='tomato',
        label=f'Top {TOP_K} selected')
plt.xlabel('Patch Index')
plt.ylabel('Attention Weight')
plt.title(f'ABMIL Attention Weights — {VIS_PATIENT}\nRed = sent to U-Net')
plt.legend()
plt.tight_layout()
plt.savefig('abmil_attention_weights.png', dpi=150)
plt.show()

# ── Plot 2: MRI modalities + GT + Prediction ──────────────────────────────
colors_mask = ['lightgray', 'blue', 'red', 'green', 'yellow']
cmap_mask   = matplotlib.colors.ListedColormap(colors_mask)
legend_names = ["Background", "ET", "NETC", "SNFH", "RC"]
modalities   = ['T1c', 'T1n', 'T2f', 'T2w']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i in range(4):
    r, c = divmod(i, 2)
    # Only use first 4 subplots (2x2 area), last 2 for GT and pred
    pass

# Row 0: T1c | T1n | T2f
for col, (mod_i, mod_name) in enumerate(zip([0, 1, 2], ['T1c', 'T1n', 'T2f'])):
    axes[0, col].imshow(volume[:, :, SLICE_Z, mod_i], cmap='gray')
    axes[0, col].set_title(mod_name, fontsize=13)
    axes[0, col].axis('off')

# Row 1: T2w | Ground Truth | Prediction
axes[1, 0].imshow(volume[:, :, SLICE_Z, 3], cmap='gray')
axes[1, 0].set_title('T2w', fontsize=13)
axes[1, 0].axis('off')

axes[1, 1].imshow(true_vol[:, :, SLICE_Z], cmap=cmap_mask, vmin=0, vmax=4)
axes[1, 1].set_title('Ground Truth', fontsize=13)
axes[1, 1].axis('off')

axes[1, 2].imshow(pred_vol[:, :, SLICE_Z], cmap=cmap_mask, vmin=0, vmax=4)
axes[1, 2].set_title('ABMIL + U-Net Prediction', fontsize=13)
axes[1, 2].axis('off')

patches_legend = [mpatches.Patch(color=colors_mask[i], label=legend_names[i])
                  for i in range(5)]
fig.legend(handles=patches_legend, loc='upper center',
           bbox_to_anchor=(0.5, 1.04), fontsize=11, ncol=5)

plt.suptitle(f'Patient: {VIS_PATIENT} | Slice Z={SLICE_Z}', fontsize=13, y=1.07)
plt.tight_layout()
plt.savefig('abmil_unet_prediction_vis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved visualisations.")


## Cell 14 — Sanity Check: Unique Labels in Predictions vs Ground Truth

In [ ]:
unique_pred = set()
unique_true = set()

for patient_id in all_patients[:20]:   # check first 20 patients only
    try:
        pred_vol, true_vol, _, _, _ = predict_patient_abmil_unet(
            root_dir, patient_id,
            abmil_model, unet_model,
            patch_size=PATCH_SIZE, top_k=TOP_K, device=DEVICE
        )
        unique_pred.update(np.unique(pred_vol).tolist())
        unique_true.update(np.unique(true_vol).tolist())
        del pred_vol, true_vol
        gc.collect()
    except Exception as e:
        print(f"Skipping {patient_id}: {e}")

print("Unique predicted labels :", sorted(unique_pred))
print("Unique ground-truth labels:", sorted(unique_true))
